# 01 · Agent core components

*Book: **AI Agents in Action, Second Edition** (Manning), chapter 2. Notebook status: in progress.*

This notebook is my record of working through the chapter: a summary of the ideas in my own words, then each exercise run step by step, with my notes on what I changed and what I learned.

The code is adapted from the book's sample repository, [AI-Agent-Workflows](https://github.com/cxbxmxcx/AI-Agent-Workflows) (Apache License 2.0). See the Credit section of the [README](README.md).

## Chapter summary

> **DRAFT: rewrite each point in my own words.** This outline comes from the code in the chapter's examples, not from the book text. I'm replacing it as I read.

An agent in this chapter is built from a few parts, and each exercise adds one:

1. **Instructions and input.** The instructions are the agent's standing rules (its role and constraints). The input is the specific request for one run.
2. **Model and model settings.** Which model answers, and how: `temperature` (repeatability), `max_tokens` (length limit), `top_p`, and the two penalties (repetition).
3. **Typed output.** Instead of free text, the agent's answer is forced into a fixed shape (a Pydantic class), so code can use it directly. The shape must be fixed: lists of small objects work, dictionaries with open-ended keys don't.
4. **Tracing.** Each run is recorded so I can inspect the model calls, tool calls and timing afterwards.
5. **Tools.** A Python function becomes a tool with `@function_tool` and is *registered* by listing it in `tools=[...]`. The model asks for it; the SDK runs it and returns the result.

**Questions I still have:**

- _add as they come up_

## Setup

Run the cells in order. The notebook needs an `OPENAI_API_KEY` in a `.env` file (here or in the repository root), and the packages from `requirements.txt` installed in the notebook's kernel.

In [ ]:
# Adapted from chapter_02/ in https://github.com/cxbxmxcx/AI-Agent-Workflows (Apache License 2.0).
# Modified: combined the chapter's examples into one notebook. A notebook already runs an
# event loop, so `await Runner.run(...)` replaces `Runner.run_sync(...)`. The variable `input`
# is renamed `user_input` (it hid Python's built-in). .env is searched for upward from here.

import os
from typing import List

from agents import Agent, ModelSettings, Runner, function_tool, trace
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ConfigDict
from typing_extensions import TypedDict

load_dotenv(find_dotenv(usecwd=True))
print("API key found:", bool(os.environ.get("OPENAI_API_KEY")))  # never print the key itself

## Exercise 1: A first agent

*Upstream: `01_first_agent.py`*

The smallest agent: instructions and an input, no tools, plain-text output. The instructions set the role and one hard constraint (five tasks, five words or fewer each).

*The output below is from my own run on 2026-10-27.*

In [ ]:
# Agent Instructions
instructions = """
You are a research planning assistant.

**TASK INSTRUCTIONS**
- You will be given a research topic.
- Your task is to provide a plan on how to research this topic.
- Output 5 concise tasks (5 words or less) to your plan.
"""

agent = Agent(
    name="Research Planner",
    instructions=instructions,
)

user_input = "learn about AI agents"

result = await Runner.run(agent, input=user_input)
print(result.final_output)

1. Survey foundational AI agents literature
2. Compare popular agentic frameworks
3. Study real-world agent applications
4. Experiment with open-source agents
5. Follow current research and benchmarks


**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 2: Model settings

*Upstream: `02_setting_agent_model_parameters.py`*

Same agent, but now I choose the model and tune how it answers. A temperature of `0.0` makes the output as repeatable as possible, `max_tokens` caps the length, and the penalties discourage repetition.

The book names the model `gpt-4.1`. If my account can't use it, I change the name.

In [ ]:
agent = Agent(
    name="Research Planner",
    instructions=instructions,
    model="gpt-4.1",  # Specify the model to use
    model_settings=ModelSettings(
        temperature=0.0,  # Set the temperature for repeatability
        max_tokens=150,  # Set the maximum number of tokens in the response
        top_p=1.0,  # Set the top-p sampling parameter
        frequency_penalty=0.5,  # Set the frequency penalty
        presence_penalty=0.5,  # Set the presence penalty
    ),
)

result = await Runner.run(agent, input=user_input)
print(result.final_output)

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 3: Typed output, a list of strings

*Upstream: `03_output_types_basic.py`*

`output_type` forces the answer into a fixed shape. Here it's an object with one field, `tasks`, a list of text items. The result is now a Python object, so I can use `result.final_output.tasks[0]` instead of cutting up a string.

In [ ]:
class ResearchPlanModel(BaseModel):
    tasks: List[str]
    """A list of tasks to perform for research."""


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
)

result = await Runner.run(agent, input=user_input)
print(result.final_output)
print("First task:", result.final_output.tasks[0])

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 4: A shape that doesn't work (dictionary)

*Upstream: `04_output_types.py`*

I want numbered tasks, so I try a dictionary: `{1: "...", 2: "..."}`. The catch is that the model's structured output needs a **fixed shape** with named fields, and a dictionary has open-ended keys. I expect this to fail, which is the point of the example.

I wrapped the call in `try/except` so the notebook keeps running and the error message is printed here.

In [ ]:
class ResearchPlanModel(BaseModel):
    tasks: dict[int, str]
    """A list of tasks to perform for research."""


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
)

try:
    result = await Runner.run(agent, input=user_input)
    print(result.final_output)
except Exception as error:  # added so "Run All" doesn't stop here
    print(type(error).__name__, ":", error)

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 5: The fix, a list of small objects

*Upstream: `05_output_types_fixed.py`*

Instead of a dictionary, a list where each item is a `Task` with two named fields. The numbering lives inside each item (`id`), and the shape is fixed. `extra='forbid'` means no other fields are allowed, which the strict mode requires.

In [ ]:
class Task(TypedDict):
    id: int
    description: str


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""

    model_config = ConfigDict(extra='forbid')


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
)

result = await Runner.run(agent, input=user_input)
for task in result.final_output.tasks:
    print(task["id"], task["description"])

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 6: Tracing

*Upstream: `06_agent_with_tracing.py`*

The Agents SDK records traces by default. `trace("name")` gives a run a label and groups everything inside the `with` block. To look at it, open the Traces page in the OpenAI dashboard ([platform.openai.com/traces](https://platform.openai.com/traces)) and find "Deep Research Workflow".

Traces contain prompts and outputs, so I check any screenshot before sharing it.

In [ ]:
class Task(TypedDict):
    id: int
    description: str


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""

    model_config = ConfigDict(extra='forbid')


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
)

with trace("Deep Research Workflow"):
    result = await Runner.run(agent, input=user_input)

for task in result.final_output.tasks:
    print(task["id"], task["description"])

*Screenshot of the trace: add one here once I've looked at it in the dashboard.*

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 7: A tool

*Upstream: `07_agent_with_tool.py`*

`@function_tool` turns a Python function into something the SDK can describe to the model (name, docstring, inputs). **Registering** the tool means listing it in `tools=[...]`. The model then asks for it, the SDK runs it, and the result goes back to the model. The model never runs my code itself.

The docstring matters, because the model reads it to decide when to use the tool.

In [ ]:
instructions = """
You are a research planning assistant.

**TASK INSTRUCTIONS**
- You will be given a research topic.
- Begin by using the tool get_research_sources()
to get a list of available research sources.
- Constrain your research plan
only to use the available research sources.
- Your task is to provide a plan for researching this topic.
- Output 5 concise tasks and specify which of the
available research sources will be used for each task.
"""


class Task(TypedDict):
    step: int
    """Task Step."""
    research_source: str
    """The source to search."""
    description: str
    """Task description."""


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""
    model_config = ConfigDict(extra='forbid')


@function_tool
def get_research_sources() -> list[str]:
    """Provides a list of research sources."""
    search_sources = [
        "Wikipedia",
        "Google",
        "YouTube",
    ]
    return search_sources


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
    tools=[get_research_sources],  # Register the tool with the agent
)

result = await Runner.run(agent, input=user_input)
for task in result.final_output.tasks:
    print(task["step"], task["research_source"], "-", task["description"])

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 7x: A tool that returns structured data

*Upstream: `07x_agent_with_tool_extended.py`*

The same idea, but the tool returns typed objects (`ResearchSource`) instead of plain strings, and the output type nests one typed object inside another.

In [ ]:
instructions = """
You are a research planning assistant.

**TASK INSTRUCTIONS**
- You will be given a research topic.
- Begin by using the tool get_research_sources() to get a list of available research sources.
- Constrain your research plan to only use the available research sources.
- Your task is to provide a plan on how to research this topic.
- Output 5 concise tasks, and specify which available research source will be used for each task.
"""


class ResearchSource(TypedDict):
    name: str
    """Name of the search source."""


class Task(TypedDict):
    step: int
    """Task Step."""
    research_source: ResearchSource
    """The source to search."""
    description: str
    """Task description."""


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""

    model_config = ConfigDict(extra='forbid')


@function_tool
def get_research_sources() -> list[ResearchSource]:
    """Provides a list of research sources."""
    search_sources = [
        ResearchSource(name="Wikipedia"),
        ResearchSource(name="Google"),
        ResearchSource(name="YouTube"),
    ]
    return search_sources


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
    tools=[get_research_sources],  # Register the tool with the agent
)

result = await Runner.run(agent, input=user_input)
for task in result.final_output.tasks:
    print(task["step"], task["research_source"]["name"], "-", task["description"])

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Exercise 8: Tools and tracing together

*Upstream: `08_agent_with_tools_tracing.py`*

Two tools, a nested output type, and a named trace. In the dashboard trace, I can see each tool call (`get_research_sources`, then `get_resource_url`) with the inputs the model chose and what the tool returned.

In [ ]:
instructions = """
You are a research planning assistant.

**TASK INSTRUCTIONS**
- You will be given a research topic.
- Begin by using the tool get_research_sources()
to get a list of available research sources.
- Use the tool get_resource_url() to look up the url for the research source.
- Constrain your research plan
only to use the available research sources.
- Your task is to provide a plan for researching this topic.
- Output 5 concise tasks and specify which of the
available research sources will be used for each task.
"""


class ResearchSource(TypedDict):
    name: str
    """Name of the search source."""
    url: str
    """URL of the search source."""


class Task(TypedDict):
    step: int
    """Task Step."""
    research_source: ResearchSource
    """The source to search."""
    description: str
    """Task description."""


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""
    model_config = ConfigDict(extra='forbid')


@function_tool
def get_research_sources() -> list[str]:
    """Provides a list of research sources."""
    search_sources = [
        "Wikipedia",
        "Google",
        "YouTube",
    ]
    return search_sources


@function_tool
def get_resource_url(research_source: str) -> str:
    """Provides a url for the research source."""
    search_sources = {
        "Wikipedia": "https://www.wikipedia.org",
        "Google": "https://www.google.com",
        "YouTube": "https://www.youtube.com",
    }
    return search_sources[research_source]


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    output_type=ResearchPlanModel,
    tools=[get_research_sources, get_resource_url],
)

with trace("Deep Research Workflow (Tools)"):
    result = await Runner.run(agent, input=user_input)

for task in result.final_output.tasks:
    source = task["research_source"]
    print(task["step"], source["name"], source["url"], "-", task["description"])

*Screenshot of the trace showing the tool calls: add one here.*

**My notes**

- **What I changed:**
- **What I noticed:**
- **What I'd try next:**

## Chapter takeaways

_Write 3 to 5 sentences in my own words: what an agent is made of, what typed outputs and tools add, and what tracing is for._

## Next

Chapter 3: MCP servers and agents.